# Assignment 2 — MapReduce: Number of Students in Each Grade Category

**Objective:** Apply the MapReduce framework to calculate the number of students in each category **S, A, B, C, D, E and F** for all subjects.

## Dataset
An official previous-semester result dataset was not provided by the instructor. Therefore, this assignment uses a **synthetic/anonymized previous-semester-style final exam dataset** containing 30 students and 5 subjects (150 exam records).

**Important:** This dataset is for demonstrating the MapReduce framework and is not claimed to be an actual university result dataset.

## 1. Load the dataset

In [2]:
import pandas as pd

# In Google Colab, upload previous_semester_final_exam_results.csv
# using the Files (folder) icon, or run the upload cell below.

from google.colab import files
uploaded = files.upload()

file_name = "previous_semester_final_exam_results.csv"
df = pd.read_csv("previous_semester_final_exam_results.csv")

print("Dataset shape:", df.shape)
display(df.head(10))

ModuleNotFoundError: No module named 'google'

## 2. Check and clean the data

In [ ]:
valid_grades = ["S", "A", "B", "C", "D", "E", "F"]

df["Student_ID"] = df["Student_ID"].astype(str).str.strip()
df["Subject"] = df["Subject"].astype(str).str.strip()
df["Grade"] = df["Grade"].astype(str).str.strip().str.upper()

# Keep only valid grade categories
df = df[df["Grade"].isin(valid_grades)].copy()

print("Students:", df["Student_ID"].nunique())
print("Subjects:", df["Subject"].nunique())
print("Exam records:", len(df))
print("Grade categories:", sorted(df["Grade"].unique()))
display(df.head())

## 3. Map phase

The Mapper converts each exam record into:

**(Subject, Grade) → 1**

For example:

`(Data Structures, A) → 1`

In [ ]:
mapped = [
    ((row.Subject, row.Grade), 1)
    for row in df.itertuples(index=False)
]

print("First 15 mapper outputs:")
for item in mapped[:15]:
    print(item)

## 4. Reduce phase

The Reducer groups identical `(Subject, Grade)` keys and adds their values.

**(Subject, Grade) → total number of students**

In [ ]:
reduced = {}

for key, value in mapped:
    reduced[key] = reduced.get(key, 0) + value

print("Reducer output:")
for key in sorted(reduced):
    print(key, "->", reduced[key])

## 5. Final result: number of students in each category for every subject

In [ ]:
result = pd.DataFrame(
    [
        {"Subject": subject, "Grade": grade, "Student_Count": count}
        for (subject, grade), count in reduced.items()
    ]
)

pivot_result = (
    result
    .pivot(index="Subject", columns="Grade", values="Student_Count")
    .fillna(0)
    .astype(int)
    .reindex(columns=valid_grades, fill_value=0)
)

display(pivot_result)

## 6. Overall grade count across all subjects

In [ ]:
overall = (
    df["Grade"]
    .value_counts()
    .reindex(valid_grades, fill_value=0)
    .rename_axis("Grade")
    .reset_index(name="Student_Count")
)

display(overall)

## 7. Verification

Each subject should contain 30 students because the synthetic dataset has 30 students taking every subject.

In [ ]:
subject_totals = df.groupby("Subject")["Student_ID"].nunique()

print("Students per subject:")
display(subject_totals.to_frame("Number_of_Students"))

assert (subject_totals == 30).all()
print("Verification passed: every subject has 30 students.")

## Conclusion

The MapReduce framework was successfully implemented to count students in grade categories S, A, B, C, D, E and F for every subject. The Mapper emitted `(Subject, Grade) → 1`, and the Reducer summed the values for each key. The final table provides the grade distribution subject-wise.

**Dataset note:** Since no official previous-semester result dataset was provided, a synthetic/anonymized dataset was used for demonstration.